# B2 — KG Enrichment

Runs SPARQL UPDATE queries to enrich the knowledge graph with:
1. Position sequences (`trajecto:hasNextPosition`)
2. Interaction classification (Passing / Meaningful / Immersive)
3. Start/Destination interactions

No temporal context enrichment (BIRD has elapsed timestamps, not wall-clock).
No Wikidata linking or holiday annotation.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('helpers'))
from fuseki_config import query_endpoint, update_endpoint
from fuseki_queries import execute_sparql_query, execute_sparql_update, format_results

In [ ]:
PREFIXES = """
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX poi:      <http://w3id.org/trajecto/poi#>
PREFIX dul:      <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>
PREFIX datacron: <http://www.datacron-project.eu/datAcron#>
PREFIX rdf:      <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd:      <http://www.w3.org/2001/XMLSchema#>
"""

## 1. Position Sequences

Link consecutive positions within each trajectory via `trajecto:hasNextPosition`.

In [ ]:
# Get all positions ordered by trajectory and timestamp
query = PREFIXES + """
SELECT ?traj ?pos ?timestamp
WHERE {
    ?pos dul:isPartOf ?traj .
    ?pos trajecto:timestamp ?timestamp .
}
ORDER BY ?traj ?timestamp
"""
result = execute_sparql_query(query_endpoint, query)
df = format_results(result)
print(f'Total positions: {len(df)}')

# Build hasNextPosition triples
insert_lines = []
prev_traj = None
prev_pos = None

for _, row in df.iterrows():
    if row['traj'] == prev_traj and prev_pos:
        insert_lines.append(f'  <{prev_pos}> trajecto:hasNextPosition <{row["pos"]}> .')
    prev_traj = row['traj']
    prev_pos = row['pos']

# Batch insert
batch_size = 1000
total = 0
for i in range(0, len(insert_lines), batch_size):
    batch = insert_lines[i:i+batch_size]
    triples = '\n'.join(batch)
    update = PREFIXES + f"""INSERT DATA {{\n{triples}\n}}"""
    if execute_sparql_update(update_endpoint, update):
        total += len(batch)

print(f'Inserted {total} hasNextPosition links')

## 2. Interaction Classification

Based on dwell time thresholds (calibrated to museum dwell distribution — median 9s, P90=34s, P95=50s):
- **Passing**: < 15s
- **Meaningful**: 15–90s
- **Immersive**: > 90s (genuine deep engagement, 1.5+ minutes)

Note: ImmersiveInteraction is a subclass of MeaningfulInteraction in the ontology.

In [ ]:
# Passing: dwell < 15s
passing_query = PREFIXES + """
INSERT {
    ?interaction a poi:PassingInteraction .
}
WHERE {
    ?interaction a poi:POIInteraction .
    ?interaction poi:hasDwellTimeInSeconds ?dwell .
    FILTER(?dwell < 15.0)
    FILTER NOT EXISTS { ?interaction a poi:PassingInteraction }
}
"""
execute_sparql_update(update_endpoint, passing_query)

# Meaningful: 15s <= dwell <= 90s
meaningful_query = PREFIXES + """
INSERT {
    ?interaction a poi:MeaningfulInteraction .
}
WHERE {
    ?interaction a poi:POIInteraction .
    ?interaction poi:hasDwellTimeInSeconds ?dwell .
    FILTER(?dwell >= 15.0 && ?dwell <= 90.0)
    FILTER NOT EXISTS { ?interaction a poi:MeaningfulInteraction }
}
"""
execute_sparql_update(update_endpoint, meaningful_query)

# Immersive: dwell > 90s (also MeaningfulInteraction via ontology subclass)
immersive_query = PREFIXES + """
INSERT {
    ?interaction a poi:ImmersiveInteraction .
    ?interaction a poi:MeaningfulInteraction .
}
WHERE {
    ?interaction a poi:POIInteraction .
    ?interaction poi:hasDwellTimeInSeconds ?dwell .
    FILTER(?dwell > 90.0)
    FILTER NOT EXISTS { ?interaction a poi:ImmersiveInteraction }
}
"""
execute_sparql_update(update_endpoint, immersive_query)

print('Interaction classification complete')

In [ ]:
# Verify counts
for int_type in ['PassingInteraction', 'MeaningfulInteraction', 'ImmersiveInteraction']:
    q = f"""
    SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE {{
        ?i a <http://w3id.org/trajecto/poi#{int_type}> .
    }}
    """
    result = execute_sparql_query(query_endpoint, q)
    df = format_results(result)
    count = df.iloc[0]['c'] if not df.empty else '?'
    print(f'  {int_type}: {count}')

## 3. Start & Destination Interactions

First and last observed artworks per trajectory.

In [ ]:
# Start interaction: earliest interaction per trajectory
start_query = PREFIXES + """
INSERT {
    ?interaction a poi:StartInteraction .
}
WHERE {
    {
        SELECT ?traj (MIN(?ts) AS ?minTs)
        WHERE {
            ?pos dul:isPartOf ?traj .
            ?pos trajecto:hasInteraction ?int .
            ?pos trajecto:timestamp ?ts .
        }
        GROUP BY ?traj
    }
    ?pos dul:isPartOf ?traj .
    ?pos trajecto:timestamp ?minTs .
    ?pos trajecto:hasInteraction ?interaction .
}
"""
execute_sparql_update(update_endpoint, start_query)

# Destination interaction: latest interaction per trajectory
dest_query = PREFIXES + """
INSERT {
    ?interaction a poi:DestinationInteraction .
}
WHERE {
    {
        SELECT ?traj (MAX(?ts) AS ?maxTs)
        WHERE {
            ?pos dul:isPartOf ?traj .
            ?pos trajecto:hasInteraction ?int .
            ?pos trajecto:timestamp ?ts .
        }
        GROUP BY ?traj
    }
    ?pos dul:isPartOf ?traj .
    ?pos trajecto:timestamp ?maxTs .
    ?pos trajecto:hasInteraction ?interaction .
}
"""
execute_sparql_update(update_endpoint, dest_query)

# Verify
for int_type in ['StartInteraction', 'DestinationInteraction']:
    q = f"""
    SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE {{
        ?i a <http://w3id.org/trajecto/poi#{int_type}> .
    }}
    """
    result = execute_sparql_query(query_endpoint, q)
    df = format_results(result)
    count = df.iloc[0]['c'] if not df.empty else '?'
    print(f'  {int_type}: {count}')

## 4. Assign Trajectories to Users

In [ ]:
# Link trajectories to users via user:belongsToUser
user_link_query = PREFIXES + """
PREFIX user: <http://w3id.org/trajecto/user#>

INSERT {
    ?traj user:belongsToUser ?person .
}
WHERE {
    ?person a trajecto:Person .
    ?person trajecto:formsTrajectory ?traj .
    FILTER NOT EXISTS { ?traj user:belongsToUser ?any }
}
"""
execute_sparql_update(update_endpoint, user_link_query)

# Verify
q = """SELECT (COUNT(*) AS ?c) WHERE { ?t <http://w3id.org/trajecto/user#belongsToUser> ?u }"""
result = execute_sparql_query(query_endpoint, q)
df = format_results(result)
print(f'Trajectory-user links: {df.iloc[0]["c"] if not df.empty else "?"}')

## 5. Enrichment Summary

In [ ]:
summary_queries = [
    ('hasNextPosition links', 'SELECT (COUNT(*) AS ?c) WHERE { ?s <http://w3id.org/trajecto#hasNextPosition> ?o }'),
    ('PassingInteraction', 'SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE { ?i a <http://w3id.org/trajecto/poi#PassingInteraction> }'),
    ('MeaningfulInteraction', 'SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE { ?i a <http://w3id.org/trajecto/poi#MeaningfulInteraction> }'),
    ('ImmersiveInteraction', 'SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE { ?i a <http://w3id.org/trajecto/poi#ImmersiveInteraction> }'),
    ('StartInteraction', 'SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE { ?i a <http://w3id.org/trajecto/poi#StartInteraction> }'),
    ('DestinationInteraction', 'SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE { ?i a <http://w3id.org/trajecto/poi#DestinationInteraction> }'),
    ('User-trajectory links', 'SELECT (COUNT(*) AS ?c) WHERE { ?t <http://w3id.org/trajecto/user#belongsToUser> ?u }'),
    ('Total triples', 'SELECT (COUNT(*) AS ?c) WHERE { ?s ?p ?o }'),
]

print('=== Enrichment Summary ===')
for label, query in summary_queries:
    result = execute_sparql_query(query_endpoint, query)
    df = format_results(result)
    count = df.iloc[0]['c'] if not df.empty else '?'
    print(f'  {label}: {count}')